In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

# Analyzing the trained models
> Access and interpret the models after fitting

## Data setup

In [ ]:
from mlforecast.utils import generate_daily_series

In [ ]:
series = generate_daily_series(10)
series.head()

## Training

Suppose that you want to train a linear regression model using the day of the week and lag1 as features.

In [ ]:
from sklearn.linear_model import LinearRegression

from mlforecast import MLForecast

In [ ]:
fcst = MLForecast(
    freq='D',
    models={'lr': LinearRegression()},
    lags=[1],
    date_features=['dayofweek'],
)

In [ ]:
fcst.fit(series)

What `MLForecast.fit` does is save the required data for the predict step and also train the models (in this case the linear regression). The trained models are available in the `MLForecast.models_` attribute, which is a dictionary where the keys are the model names and the values are the model themselves.

In [ ]:
fcst.models_

## Inspect parameters

We can access the linear regression coefficients in the following way:

In [ ]:
fcst.models_['lr'].intercept_, fcst.models_['lr'].coef_

## SHAP

In [ ]:
import shap

### Training set

If you need to generate the training data you can use `MLForecast.preprocess`.

In [ ]:
prep = fcst.preprocess(series)
prep.head()

We extract the X, which involves dropping the info columns (id + times) and the target

In [ ]:
X = prep.drop(columns=['unique_id', 'ds', 'y'])
X.head()

We can now compute the shap values

In [ ]:
X100 = shap.utils.sample(X, 100)
explainer = shap.Explainer(fcst.models_['lr'].predict, X100)
shap_values = explainer(X)

And visualize them

In [ ]:
shap.plots.beeswarm(shap_values)

### Predictions

Sometimes you want to determine why the model gave a specific prediction. In order to do this you need the input features, which aren't returned by default, but you can retrieve them using a callback.

In [ ]:
from mlforecast.callbacks import SaveFeatures

In [ ]:
save_feats = SaveFeatures()
preds = fcst.predict(1, before_predict_callback=save_feats)
preds.head()

You can now retrieve the features by using `SaveFeatures.get_features`

In [ ]:
features = save_feats.get_features()
features.head()

And use those features to compute the shap values.

In [ ]:
shap_values_predictions = explainer(features)

We can now analyze what influenced the prediction for `'id_4'`.

In [ ]:
round(preds.loc[4, 'lr'], 3)

In [ ]:
shap.plots.waterfall(shap_values_predictions[4])